# Notebook 12: Training Loop Explained - From Scratch

## 📚 Historical Context

**Timeline**: 2015-2023 - Evolution from manual training to high-level APIs

**The Journey of Training APIs**:
- **2015-2016**: Pure PyTorch - Manual everything
  - Write your own training loops
  - Manual gradient accumulation
  - Manual learning rate scheduling
  - Pain point: Boilerplate code everywhere

- **2018**: PyTorch-Lightning emerges
  - Automates distributed training
  - Handles device placement
  - Still requires understanding fundamentals

- **2019**: HuggingFace Trainer API
  - `trainer.train()` - that's it!
  - Handles checkpointing, logging, evaluation
  - Hides complexity... but also understanding

- **2023+**: Era of abstraction
  - Most practitioners never write raw training loops
  - Great for productivity, bad for debugging
  - When things break, you're lost without fundamentals

**Why Learn This?**:
- **Debugging**: When Trainer API fails, you need to understand what's happening
- **Customization**: Custom loss functions, specialized gradient updates
- **Research**: New training techniques require raw loops
- **Interviews**: "Implement training loop from scratch" is common

## 🎯 What You'll Learn

1. **Basic Training Loop** - Forward, backward, optimizer step
2. **Gradient Accumulation** - Train with limited memory
3. **Learning Rate Scheduling** - Warmup, decay, cosine annealing
4. **Checkpointing** - Save/resume training
5. **Logging** - wandb integration
6. **Early Stopping** - Prevent overfitting
7. **Production Loop** - All features combined

## 💡 Why This Matters

Understanding the training loop is CRITICAL because:
- **Memory issues?** → Likely batch size or gradient accumulation
- **Not converging?** → Check LR schedule or loss computation
- **Training crashes?** → Checkpointing saves your work
- **Metrics weird?** → Logging reveals the issue

**Real-world scenario**: You're fine-tuning Llama-7B but keep running OOM (out of memory). The Trainer API doesn't help. With manual loop knowledge, you can implement gradient accumulation, mixed precision, and gradient checkpointing yourself.

---

In [ ]:
# Import libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import (
    CosineAnnealingLR, 
    LinearLR, 
    SequentialLR,
    ReduceLROnPlateau
)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import time
import os
from pathlib import Path
import json
from dataclasses import dataclass, asdict

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Part 1: Simple Training Loop - The Basics

### The Core Training Loop:

```python
for epoch in range(num_epochs):
    for batch in dataloader:
        # 1. Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # 2. Backward pass
        optimizer.zero_grad()  # Clear old gradients
        loss.backward()        # Compute gradients
        
        # 3. Optimizer step
        optimizer.step()       # Update weights
```

### What Happens Under the Hood:

**1. Forward Pass**:
```
Input → Layer 1 → Activation → Layer 2 → ... → Output
PyTorch builds computation graph automatically
```

**2. Backward Pass (loss.backward())**:
```
Output ← ∂Loss/∂Output ← Layer 2 ← Layer 1 ← Input
Chain rule applied automatically via autograd
Gradients accumulated in parameter.grad
```

**3. Optimizer Step**:
```
For each parameter:
    parameter = parameter - learning_rate * parameter.grad
    
Different optimizers have different update rules:
- SGD: Simple gradient descent
- Adam: Adaptive learning rates + momentum
- AdamW: Adam + weight decay (best for transformers)
```

### Why zero_grad()?
PyTorch **accumulates** gradients by default. If you don't zero them, you get:
```
Iteration 1: grad = ∂Loss1/∂θ
Iteration 2: grad = ∂Loss1/∂θ + ∂Loss2/∂θ  ← WRONG!
```

This is actually useful for gradient accumulation (we'll see later).

In [ ]:
# Simple model and dataset for demonstration
class SimpleClassifier(nn.Module):
    """Simple 2-layer classifier for demonstration."""
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.1)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

class SimpleDataset(Dataset):
    """Generate synthetic classification data."""
    def __init__(self, num_samples, input_dim, num_classes):
        self.num_samples = num_samples
        self.input_dim = input_dim
        self.num_classes = num_classes
        
        # Generate random data
        self.data = torch.randn(num_samples, input_dim)
        self.labels = torch.randint(0, num_classes, (num_samples,))
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

# Create dataset and dataloader
train_dataset = SimpleDataset(num_samples=1000, input_dim=128, num_classes=5)
val_dataset = SimpleDataset(num_samples=200, input_dim=128, num_classes=5)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Initialize model
model = SimpleClassifier(input_dim=128, hidden_dim=256, num_classes=5).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def train_basic_loop(model, train_loader, val_loader, num_epochs=5):
    """
    Basic training loop - no frills.
    
    This is the FOUNDATION. Understand this before moving to advanced features.
    """
    # Setup
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=1e-3)
    
    # Track metrics
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    
    for epoch in range(num_epochs):
        # ========== TRAINING ==========
        model.train()  # Set model to training mode (enables dropout, etc.)
        train_loss = 0.0
        
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            # Move to device
            inputs, labels = inputs.to(device), labels.to(device)
            
            # Step 1: Forward pass
            # Compute predictions
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            # Step 2: Backward pass
            # Clear old gradients from previous batch
            optimizer.zero_grad()
            
            # Compute gradients via backpropagation
            loss.backward()
            
            # Step 3: Update weights
            optimizer.step()
            
            # Track loss
            train_loss += loss.item()
        
        # Average loss over epoch
        train_loss /= len(train_loader)
        
        # ========== VALIDATION ==========
        model.eval()  # Set to evaluation mode (disables dropout, etc.)
        val_loss = 0.0
        correct = 0
        total = 0
        
        # Don't compute gradients during validation
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                
                # Compute accuracy
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        val_loss /= len(val_loader)
        val_acc = correct / total
        
        # Store metrics
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}/{num_epochs}:")
        print(f"  Train Loss: {train_loss:.4f}")
        print(f"  Val Loss:   {val_loss:.4f}")
        print(f"  Val Acc:    {val_acc:.4f}")
    
    return history

# Train the model
print("="*60)
print("BASIC TRAINING LOOP")
print("="*60)
history = train_basic_loop(model, train_loader, val_loader, num_epochs=5)

# Plot results
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Over Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['val_acc'], label='Val Accuracy', marker='o', color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 2: Gradient Accumulation

### The Problem:
You want batch_size=128, but only have 8GB GPU → OOM error!

### The Solution:
**Gradient Accumulation** - Simulate large batches with multiple small batches.

### How It Works:
```
Normal training (batch_size=128):
  1. Forward pass on 128 samples
  2. Backward pass
  3. Update weights

Gradient accumulation (batch_size=32, accumulation_steps=4):
  1. Forward on 32 samples, backward → gradients stored
  2. Forward on 32 samples, backward → gradients accumulated
  3. Forward on 32 samples, backward → gradients accumulated
  4. Forward on 32 samples, backward → gradients accumulated
  5. Update weights using accumulated gradients
  6. Zero gradients
```

### Effective Batch Size:
```
effective_batch_size = batch_size * accumulation_steps
Example: 32 * 4 = 128
```

### Key Implementation Detail:
Must scale loss by accumulation steps to get correct gradient magnitude:
```python
loss = loss / accumulation_steps
```

### When to Use:
- GPU memory limited
- Want large batch sizes for stability
- Training large models (Llama-7B, 13B, etc.)

### Trade-off:
- Pro: Train with any batch size regardless of memory
- Con: Slower training (more forward/backward passes)

In [ ]:
def train_with_gradient_accumulation(
    model, 
    train_loader, 
    val_loader, 
    num_epochs=5,
    accumulation_steps=4
):
    """
    Training loop with gradient accumulation.
    
    Simulates larger batch sizes by accumulating gradients over multiple
    forward/backward passes before updating weights.
    
    Args:
        accumulation_steps: Number of steps to accumulate gradients
                           Effective batch = batch_size * accumulation_steps
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=1e-3)
    
    history = {'train_loss': [], 'val_loss': []}
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        
        # Important: Only zero gradients at the start
        optimizer.zero_grad()
        
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            
            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            # Scale loss by accumulation steps
            # This ensures gradient magnitude is correct
            loss = loss / accumulation_steps
            
            # Backward pass (gradients accumulate)
            loss.backward()
            
            # Only update weights every accumulation_steps
            if (batch_idx + 1) % accumulation_steps == 0:
                # Gradient clipping (common with gradient accumulation)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                # Update weights
                optimizer.step()
                
                # Zero gradients for next accumulation cycle
                optimizer.zero_grad()
            
            train_loss += loss.item() * accumulation_steps  # Scale back for logging
        
        train_loss /= len(train_loader)
        
        # Validation (same as before)
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
        
        val_loss /= len(val_loader)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        
        print(f"Epoch {epoch+1}/{num_epochs}:")
        print(f"  Train Loss: {train_loss:.4f}")
        print(f"  Val Loss:   {val_loss:.4f}")
        print(f"  Effective batch size: {train_loader.batch_size * accumulation_steps}")
    
    return history

# Test gradient accumulation
print("\n" + "="*60)
print("TRAINING WITH GRADIENT ACCUMULATION")
print("="*60)

# Reinitialize model
model = SimpleClassifier(input_dim=128, hidden_dim=256, num_classes=5).to(device)

# Smaller batch size, but effective batch = 32 * 4 = 128
train_loader_small = DataLoader(train_dataset, batch_size=8, shuffle=True)

history_ga = train_with_gradient_accumulation(
    model, train_loader_small, val_loader, 
    num_epochs=5, accumulation_steps=4
)

print("\n" + "="*60)
print("KEY INSIGHT:")
print("Batch size 8 with 4 accumulation steps ≈ batch size 32 training")
print("But uses 4x less GPU memory!")
print("="*60)

## Part 3: Learning Rate Scheduling

### Why Learning Rate Matters:

**Fixed LR problems**:
- Too high → Doesn't converge, loss oscillates
- Too low → Converges slowly, gets stuck in local minima
- No single LR is optimal throughout training

### Common Schedules:

**1. Constant** (no schedule)
```
lr = base_lr  # Same throughout training
```

**2. Linear Warmup + Cosine Decay** (Most popular for transformers)
```
Warmup: lr increases linearly from 0 to base_lr
Decay:  lr decreases following cosine curve to 0
```
Used by: BERT, GPT-3, Llama

**3. Step Decay**
```
lr = base_lr * 0.1  every N epochs
```

**4. ReduceLROnPlateau** (Adaptive)
```
If validation loss doesn't improve for N epochs:
    lr = lr * factor
```

### Warmup - Why It's Critical:

**Problem**: Starting with high LR can cause:
- Gradient explosion in early steps
- Model diverging immediately
- Unstable training

**Solution**: Gradually increase LR in first few steps
```
Steps 0-1000:     lr increases 0 → base_lr
Steps 1000+:      lr decays base_lr → 0
```

### Typical Values for Transformers:
- Base LR: 1e-4 to 1e-3
- Warmup steps: 500-2000 (or 5-10% of total steps)
- Minimum LR: 0 or 1e-6

In [ ]:
def create_scheduler_comparison():
    """
    Visualize different learning rate schedules.
    """
    num_epochs = 50
    steps_per_epoch = 100
    total_steps = num_epochs * steps_per_epoch
    base_lr = 1e-3
    
    # Create dummy model and optimizer
    model = nn.Linear(10, 10)
    optimizer = AdamW(model.parameters(), lr=base_lr)
    
    # Different schedulers
    schedulers = {
        'Constant': None,
        'Linear Decay': LinearLR(
            optimizer, start_factor=1.0, end_factor=0.0, total_iters=total_steps
        ),
        'Cosine Annealing': CosineAnnealingLR(
            optimizer, T_max=total_steps, eta_min=0
        ),
        'Warmup + Cosine': SequentialLR(
            optimizer,
            schedulers=[
                LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=1000),
                CosineAnnealingLR(optimizer, T_max=total_steps-1000, eta_min=1e-6)
            ],
            milestones=[1000]
        )
    }
    
    # Track learning rates
    lr_histories = {name: [] for name in schedulers.keys()}
    
    # Simulate training
    for name, scheduler in schedulers.items():
        # Reset optimizer
        for param_group in optimizer.param_groups:
            param_group['lr'] = base_lr
        
        if scheduler is not None:
            scheduler = schedulers[name]  # Recreate
        
        for step in range(total_steps):
            current_lr = optimizer.param_groups[0]['lr']
            lr_histories[name].append(current_lr)
            
            if scheduler is not None:
                scheduler.step()
    
    # Plot
    plt.figure(figsize=(14, 6))
    
    for name, lrs in lr_histories.items():
        steps = np.arange(len(lrs))
        plt.plot(steps, lrs, label=name, linewidth=2, alpha=0.8)
    
    plt.xlabel('Training Step', fontsize=12)
    plt.ylabel('Learning Rate', fontsize=12)
    plt.title('Learning Rate Schedules Comparison', fontsize=14)
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    return lr_histories

print("="*60)
print("LEARNING RATE SCHEDULES")
print("="*60)
lr_histories = create_scheduler_comparison()

print("\nSchedule Characteristics:")
print("\n1. Constant:")
print("   - Simple, no tuning needed")
print("   - Often suboptimal")
print("\n2. Linear Decay:")
print("   - Gradual reduction")
print("   - Good for simple tasks")
print("\n3. Cosine Annealing:")
print("   - Smooth decay")
print("   - Popular in computer vision")
print("\n4. Warmup + Cosine:")
print("   - BEST for transformers")
print("   - Stable start + smooth decay")
print("   - Used in BERT, GPT, Llama")
print("="*60)

In [ ]:
def train_with_scheduler(
    model, 
    train_loader, 
    val_loader, 
    num_epochs=10,
    base_lr=1e-3,
    warmup_steps=100
):
    """
    Training loop with warmup + cosine annealing schedule.
    
    This is the STANDARD schedule for transformer fine-tuning.
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=base_lr)
    
    # Total steps for cosine annealing
    total_steps = len(train_loader) * num_epochs
    
    # Create warmup + cosine schedule
    warmup_scheduler = LinearLR(
        optimizer, 
        start_factor=0.01,  # Start at 1% of base_lr
        end_factor=1.0,     # Reach 100% of base_lr
        total_iters=warmup_steps
    )
    
    cosine_scheduler = CosineAnnealingLR(
        optimizer, 
        T_max=total_steps - warmup_steps,
        eta_min=1e-6  # Minimum LR at end
    )
    
    scheduler = SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, cosine_scheduler],
        milestones=[warmup_steps]
    )
    
    history = {
        'train_loss': [], 
        'val_loss': [], 
        'learning_rate': []
    }
    
    global_step = 0
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            
            # Forward + backward
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            # Step scheduler (per-step, not per-epoch!)
            current_lr = optimizer.param_groups[0]['lr']
            scheduler.step()
            
            train_loss += loss.item()
            global_step += 1
            
            # Update progress bar
            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'lr': f'{current_lr:.6f}'
            })
            
            history['learning_rate'].append(current_lr)
        
        train_loss /= len(train_loader)
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
        
        val_loss /= len(val_loader)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        
        print(f"\nEpoch {epoch+1} Summary:")
        print(f"  Train Loss: {train_loss:.4f}")
        print(f"  Val Loss:   {val_loss:.4f}")
        print(f"  Current LR: {optimizer.param_groups[0]['lr']:.6f}")
    
    return history

# Train with scheduling
print("\n" + "="*60)
print("TRAINING WITH LR SCHEDULING")
print("="*60)

model = SimpleClassifier(input_dim=128, hidden_dim=256, num_classes=5).to(device)
history_sched = train_with_scheduler(
    model, train_loader, val_loader, 
    num_epochs=10, base_lr=1e-3, warmup_steps=50
)

# Plot learning rate schedule
plt.figure(figsize=(10, 4))
plt.plot(history_sched['learning_rate'])
plt.xlabel('Training Step')
plt.ylabel('Learning Rate')
plt.title('Learning Rate Schedule During Training')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 4: Checkpointing and Resuming Training

### Why Checkpointing Matters:

**Scenarios where you MUST have checkpoints**:
1. Training crashes (OOM, hardware failure, preemption)
2. Want to test model at different epochs
3. Ensemble multiple checkpoints
4. Resume training with different hyperparameters
5. Cloud training with spot instances (can be killed anytime)

### What to Save:

**Minimal** (for inference only):
```python
torch.save(model.state_dict(), 'model.pt')
```

**Full checkpoint** (for resuming training):
```python
checkpoint = {
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'loss': loss,
    'metrics': metrics,
    'config': config  # Hyperparameters
}
torch.save(checkpoint, 'checkpoint.pt')
```

### Checkpoint Strategies:

**1. Save every N epochs**
```python
if epoch % save_every == 0:
    save_checkpoint()
```

**2. Save only best model**
```python
if val_loss < best_val_loss:
    best_val_loss = val_loss
    save_checkpoint('best_model.pt')
```

**3. Keep last K checkpoints**
```python
# Keeps checkpoints for last 3 epochs
checkpoint_files = ['ckpt_epoch_1.pt', 'ckpt_epoch_2.pt', 'ckpt_epoch_3.pt']
if len(checkpoint_files) > 3:
    os.remove(checkpoint_files[0])  # Delete oldest
```

### Storage Considerations:
- Llama-7B checkpoint: ~13GB (FP32) or ~7GB (FP16)
- Llama-70B checkpoint: ~130GB (FP32)
- Solution: Save in FP16, use cloud storage

In [ ]:
def save_checkpoint(model, optimizer, scheduler, epoch, metrics, checkpoint_dir):
    """
    Save a training checkpoint.
    
    This saves EVERYTHING needed to resume training exactly where you left off.
    """
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'metrics': metrics,
        'timestamp': time.time()
    }
    
    # Add scheduler if present
    if scheduler is not None:
        checkpoint['scheduler_state_dict'] = scheduler.state_dict()
    
    # Save checkpoint
    checkpoint_path = checkpoint_dir / f'checkpoint_epoch_{epoch}.pt'
    torch.save(checkpoint, checkpoint_path)
    print(f"Saved checkpoint: {checkpoint_path}")
    
    # Save metrics as JSON for easy reading
    metrics_path = checkpoint_dir / f'metrics_epoch_{epoch}.json'
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=2)
    
    return checkpoint_path

def load_checkpoint(checkpoint_path, model, optimizer=None, scheduler=None):
    """
    Load a training checkpoint.
    
    Args:
        checkpoint_path: Path to checkpoint file
        model: Model to load weights into
        optimizer: (Optional) Optimizer to restore state
        scheduler: (Optional) Scheduler to restore state
    
    Returns:
        epoch: Epoch number from checkpoint
        metrics: Metrics dictionary
    """
    print(f"Loading checkpoint: {checkpoint_path}")
    
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Restore model
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Restore optimizer
    if optimizer is not None and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    # Restore scheduler
    if scheduler is not None and 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    
    epoch = checkpoint['epoch']
    metrics = checkpoint.get('metrics', {})
    
    print(f"Resumed from epoch {epoch}")
    print(f"Metrics: {metrics}")
    
    return epoch, metrics

def train_with_checkpointing(
    model,
    train_loader,
    val_loader,
    num_epochs=10,
    checkpoint_dir='./checkpoints',
    save_every=2,
    resume_from=None
):
    """
    Training loop with checkpointing.
    
    Args:
        save_every: Save checkpoint every N epochs
        resume_from: Path to checkpoint to resume from
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=1e-3)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)
    
    start_epoch = 0
    best_val_loss = float('inf')
    history = {'train_loss': [], 'val_loss': []}
    
    # Resume from checkpoint if provided
    if resume_from is not None:
        start_epoch, metrics = load_checkpoint(
            resume_from, model, optimizer, scheduler
        )
        start_epoch += 1  # Start from next epoch
        if 'best_val_loss' in metrics:
            best_val_loss = metrics['best_val_loss']
    
    for epoch in range(start_epoch, num_epochs):
        # Training
        model.train()
        train_loss = 0.0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
        
        val_loss /= len(val_loader)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        
        # Update learning rate
        scheduler.step()
        
        print(f"Epoch {epoch+1}/{num_epochs}:")
        print(f"  Train Loss: {train_loss:.4f}")
        print(f"  Val Loss:   {val_loss:.4f}")
        
        # Save checkpoint periodically
        if (epoch + 1) % save_every == 0:
            metrics = {
                'train_loss': train_loss,
                'val_loss': val_loss,
                'best_val_loss': best_val_loss
            }
            save_checkpoint(
                model, optimizer, scheduler, epoch, metrics, checkpoint_dir
            )
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            metrics = {
                'train_loss': train_loss,
                'val_loss': val_loss,
                'best_val_loss': best_val_loss
            }
            best_path = save_checkpoint(
                model, optimizer, scheduler, epoch, metrics,
                Path(checkpoint_dir) / 'best'
            )
            print(f"  New best model! Val loss: {val_loss:.4f}")
    
    return history

# Test checkpointing
print("\n" + "="*60)
print("TRAINING WITH CHECKPOINTING")
print("="*60)

model = SimpleClassifier(input_dim=128, hidden_dim=256, num_classes=5).to(device)
checkpoint_dir = './demo_checkpoints'

# Train for a few epochs
history = train_with_checkpointing(
    model, train_loader, val_loader,
    num_epochs=6,
    checkpoint_dir=checkpoint_dir,
    save_every=2
)

print("\n" + "="*60)
print("Checkpoints saved! You can now:")
print("1. Resume training from any checkpoint")
print("2. Load best model for inference")
print("3. Compare models from different epochs")
print("="*60)

## Part 5: Weights & Biases (wandb) Integration

### Why Use wandb?

**Problems with print statements**:
- Lost when terminal closes
- Hard to compare multiple runs
- No visualizations
- Can't share with team

**wandb solves this**:
- Automatic logging to cloud
- Beautiful dashboards
- Compare 100s of experiments
- Track hyperparameters
- Save artifacts (models, predictions)

### What to Log:

**Metrics** (every step/epoch):
- Train/val loss
- Train/val accuracy
- Learning rate
- Gradient norms

**System** (optional):
- GPU utilization
- Memory usage
- Time per epoch

**Hyperparameters** (at start):
- Model config
- Learning rate
- Batch size
- Optimizer settings

**Artifacts** (at milestones):
- Best model checkpoint
- Predictions on validation set
- Attention visualizations

### Basic Usage:
```python
import wandb

# Initialize
wandb.init(project='my-project', config=hyperparameters)

# Log metrics
wandb.log({'train_loss': loss, 'learning_rate': lr})

# Log model
wandb.save('model.pt')

# Finish
wandb.finish()
```

**Note**: This demo uses print statements since wandb requires an account. In production, replace prints with `wandb.log()`.

In [ ]:
# Demo of wandb-style logging (without actual wandb dependency)
class WandbLogger:
    """
    Simplified logger that mimics wandb API.
    
    In production, replace this with actual wandb.init() and wandb.log().
    """
    def __init__(self, project_name, config):
        self.project_name = project_name
        self.config = config
        self.history = []
        print(f"Initialized logger for project: {project_name}")
        print(f"Config: {config}")
    
    def log(self, metrics, step=None):
        """Log metrics."""
        self.history.append(metrics)
        if step is not None:
            print(f"Step {step}: {metrics}")
    
    def finish(self):
        """Finish logging."""
        print("Logging finished.")
        print(f"Total steps logged: {len(self.history)}")

def train_with_logging(
    model,
    train_loader,
    val_loader,
    num_epochs=5,
    config=None
):
    """
    Training loop with comprehensive logging.
    
    This shows what to log and when. In production, use actual wandb.
    """
    # Default config
    if config is None:
        config = {
            'model': 'SimpleClassifier',
            'learning_rate': 1e-3,
            'batch_size': 32,
            'optimizer': 'AdamW',
            'num_epochs': num_epochs,
            'hidden_dim': 256
        }
    
    # Initialize logger (in production: wandb.init(project='...', config=config))
    logger = WandbLogger(project_name='training-demo', config=config)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=config['learning_rate'])
    
    global_step = 0
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        epoch_metrics = {
            'train_loss': 0.0,
            'train_steps': 0
        }
        
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            
            # Compute gradient norm (useful for debugging)
            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(), max_norm=1.0
            )
            
            optimizer.step()
            
            # Log every N steps
            if batch_idx % 10 == 0:
                metrics = {
                    'train/loss': loss.item(),
                    'train/grad_norm': grad_norm.item(),
                    'train/learning_rate': optimizer.param_groups[0]['lr'],
                    'epoch': epoch
                }
                logger.log(metrics, step=global_step)
            
            epoch_metrics['train_loss'] += loss.item()
            epoch_metrics['train_steps'] += 1
            global_step += 1
        
        # Average training loss
        epoch_metrics['train_loss'] /= epoch_metrics['train_steps']
        
        # Validation
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        val_loss /= len(val_loader)
        val_acc = correct / total
        
        # Log epoch summary
        epoch_summary = {
            'epoch': epoch,
            'train/epoch_loss': epoch_metrics['train_loss'],
            'val/loss': val_loss,
            'val/accuracy': val_acc,
        }
        logger.log(epoch_summary, step=global_step)
        
        print(f"\nEpoch {epoch+1}/{num_epochs}:")
        print(f"  Train Loss: {epoch_metrics['train_loss']:.4f}")
        print(f"  Val Loss:   {val_loss:.4f}")
        print(f"  Val Acc:    {val_acc:.4f}")
    
    logger.finish()
    return logger.history

# Test logging
print("\n" + "="*60)
print("TRAINING WITH LOGGING")
print("="*60)

model = SimpleClassifier(input_dim=128, hidden_dim=256, num_classes=5).to(device)
history = train_with_logging(model, train_loader, val_loader, num_epochs=3)

print("\n" + "="*60)
print("In production, this would:")
print("1. Upload metrics to wandb cloud")
print("2. Create interactive dashboards")
print("3. Allow comparing multiple runs")
print("4. Track system metrics (GPU, memory)")
print("5. Save model checkpoints as artifacts")
print("="*60)

## Part 6: Early Stopping

### The Problem:

**Overfitting progression**:
```
Epoch 1:  Train loss ↓, Val loss ↓  ← Good, learning useful patterns
Epoch 5:  Train loss ↓, Val loss ↓  ← Still good
Epoch 10: Train loss ↓, Val loss →  ← Starting to memorize
Epoch 15: Train loss ↓, Val loss ↑  ← Overfitting! Should have stopped
Epoch 20: Train loss ↓, Val loss ↑↑ ← Wasted computation
```

### Early Stopping Solution:

**Monitor validation loss**:
- If validation loss doesn't improve for N epochs → STOP
- Restore weights from best epoch
- Save time and prevent overfitting

### Key Parameters:

**patience**: Number of epochs to wait for improvement
```python
patience = 3:
  Epoch 10: val_loss = 0.5 (best so far)
  Epoch 11: val_loss = 0.51 (no improvement, counter = 1)
  Epoch 12: val_loss = 0.52 (no improvement, counter = 2)
  Epoch 13: val_loss = 0.53 (no improvement, counter = 3)
  → STOP! Restore weights from epoch 10
```

**min_delta**: Minimum improvement to count
```python
min_delta = 0.001:
  Current best: 0.500
  New val_loss: 0.499 → No improvement (difference < 0.001)
  New val_loss: 0.498 → Improvement! Reset counter
```

### Typical Values:
- patience=3-10 (depends on task stability)
- min_delta=0 or 1e-4
- mode='min' for loss, 'max' for accuracy

In [ ]:
class EarlyStopping:
    """
    Early stopping to prevent overfitting.
    
    Monitors validation metric and stops training when it stops improving.
    """
    def __init__(self, patience=5, min_delta=0, mode='min'):
        """
        Args:
            patience: Number of epochs to wait for improvement
            min_delta: Minimum change to count as improvement
            mode: 'min' for loss, 'max' for accuracy
        """
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        
        # Internal state
        self.best_value = float('inf') if mode == 'min' else float('-inf')
        self.counter = 0
        self.best_epoch = 0
        self.should_stop = False
        self.best_model_state = None
    
    def __call__(self, current_value, model, epoch):
        """
        Check if training should stop.
        
        Args:
            current_value: Current validation metric
            model: Model to save state
            epoch: Current epoch number
        
        Returns:
            should_stop: Boolean indicating if training should stop
        """
        # Check if improved
        if self.mode == 'min':
            improved = current_value < (self.best_value - self.min_delta)
        else:
            improved = current_value > (self.best_value + self.min_delta)
        
        if improved:
            # New best model!
            self.best_value = current_value
            self.best_epoch = epoch
            self.counter = 0
            
            # Save model state
            self.best_model_state = {
                k: v.cpu().clone() for k, v in model.state_dict().items()
            }
            
            print(f"  ✓ New best! {self.mode}: {current_value:.4f}")
        else:
            # No improvement
            self.counter += 1
            print(f"  ✗ No improvement (patience: {self.counter}/{self.patience})")
            
            if self.counter >= self.patience:
                self.should_stop = True
                print(f"\n⚠ Early stopping triggered!")
                print(f"   Best {self.mode}: {self.best_value:.4f} at epoch {self.best_epoch + 1}")
        
        return self.should_stop
    
    def restore_best_model(self, model):
        """Restore model to best state."""
        if self.best_model_state is not None:
            model.load_state_dict(self.best_model_state)
            print(f"Restored model from epoch {self.best_epoch + 1}")

def train_with_early_stopping(
    model,
    train_loader,
    val_loader,
    num_epochs=20,
    patience=3
):
    """
    Training loop with early stopping.
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=1e-3)
    
    # Initialize early stopping
    early_stopping = EarlyStopping(patience=patience, mode='min')
    
    history = {'train_loss': [], 'val_loss': []}
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        train_loss = 0.0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        
        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
        
        val_loss /= len(val_loader)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        
        print(f"Epoch {epoch+1}/{num_epochs}:")
        print(f"  Train Loss: {train_loss:.4f}")
        print(f"  Val Loss:   {val_loss:.4f}")
        
        # Check early stopping
        should_stop = early_stopping(val_loss, model, epoch)
        if should_stop:
            break
    
    # Restore best model
    early_stopping.restore_best_model(model)
    
    return history, early_stopping.best_epoch

# Test early stopping
print("\n" + "="*60)
print("TRAINING WITH EARLY STOPPING")
print("="*60)

model = SimpleClassifier(input_dim=128, hidden_dim=256, num_classes=5).to(device)
history, best_epoch = train_with_early_stopping(
    model, train_loader, val_loader,
    num_epochs=20,
    patience=3
)

# Plot with early stopping marker
plt.figure(figsize=(10, 5))
epochs = range(1, len(history['train_loss']) + 1)
plt.plot(epochs, history['train_loss'], label='Train Loss', marker='o')
plt.plot(epochs, history['val_loss'], label='Val Loss', marker='s')
plt.axvline(x=best_epoch + 1, color='red', linestyle='--', label=f'Best Model (Epoch {best_epoch + 1})')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training with Early Stopping')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n" + "="*60)
print(f"Training stopped early at epoch {len(history['train_loss'])}")
print(f"Best model was from epoch {best_epoch + 1}")
print(f"Saved {20 - len(history['train_loss'])} epochs of computation!")
print("="*60)

## 📊 Summary

### What We Learned:

**1. Basic Training Loop**
- Forward pass → compute loss
- Backward pass → compute gradients
- Optimizer step → update weights
- Always zero gradients between batches

**2. Gradient Accumulation**
- Simulate large batches with small GPU memory
- effective_batch_size = batch_size × accumulation_steps
- Must scale loss: `loss = loss / accumulation_steps`
- Critical for training large models (Llama, GPT)

**3. Learning Rate Scheduling**
- Fixed LR is suboptimal
- Warmup prevents early instability
- Cosine decay provides smooth convergence
- Standard for transformers: Warmup + Cosine

**4. Checkpointing**
- Save: model, optimizer, scheduler, metrics
- Strategies: periodic, best-only, keep-last-K
- Essential for long training runs
- Enables resuming after crashes

**5. Logging (wandb)**
- Track metrics, hyperparameters, system stats
- Compare experiments easily
- Share results with team
- Log every N steps, not every step

**6. Early Stopping**
- Stop when validation stops improving
- Prevents overfitting and saves time
- Typical patience: 3-10 epochs
- Always restore best model

### Production Training Loop Template:

```python
# Setup
model = load_model().to(device)
optimizer = AdamW(model.parameters(), lr=base_lr)
scheduler = create_warmup_cosine_schedule(optimizer)
early_stopping = EarlyStopping(patience=5)
wandb.init(project='my-project', config=config)

# Training loop
for epoch in range(num_epochs):
    model.train()
    
    for batch_idx, batch in enumerate(train_loader):
        # Forward
        loss = compute_loss(model, batch)
        
        # Backward with gradient accumulation
        loss = loss / accumulation_steps
        loss.backward()
        
        if (batch_idx + 1) % accumulation_steps == 0:
            # Clip gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            # Update weights
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            
            # Log
            wandb.log({
                'train/loss': loss.item(),
                'train/lr': scheduler.get_last_lr()[0]
            })
    
    # Validation
    val_metrics = validate(model, val_loader)
    wandb.log(val_metrics)
    
    # Checkpoint
    if val_metrics['val_loss'] < best_val_loss:
        save_checkpoint(model, optimizer, scheduler, epoch)
    
    # Early stopping
    if early_stopping(val_metrics['val_loss'], model, epoch):
        break

# Cleanup
early_stopping.restore_best_model(model)
wandb.finish()
```

### When to Use What:

| Scenario | Features |
|----------|----------|
| **Quick experiment** | Basic loop + early stopping |
| **Small GPU** | + Gradient accumulation |
| **Long training** | + Checkpointing |
| **Production** | + LR scheduling + wandb |
| **Large models (Llama)** | All features + mixed precision (next notebook) |

### Common Issues & Solutions:

**OOM errors**:
- ✓ Reduce batch size
- ✓ Increase gradient accumulation
- ✓ Use mixed precision (notebook 15)
- ✓ Gradient checkpointing

**Not converging**:
- ✓ Check learning rate (try 1e-4, 1e-5)
- ✓ Add warmup (500-1000 steps)
- ✓ Check gradient norms (clip if >10)
- ✓ Verify loss function is correct

**Slow training**:
- ✓ Increase batch size
- ✓ Reduce validation frequency
- ✓ Use DataLoader num_workers
- ✓ Mixed precision training

### Next Steps:

**Notebook 13: Custom Loss Functions**
- Weighted cross-entropy
- Focal loss
- Label smoothing
- Knowledge distillation
- Multi-task losses

**Notebook 15: Mixed Precision Training**
- FP16/BF16 training
- Automatic Mixed Precision (AMP)
- 2x speedup + 2x memory savings
- Critical for large models

---

**Key Takeaway**: The Trainer API is convenient, but understanding the raw training loop is ESSENTIAL for debugging, customization, and research. When your training fails at 3am and the Trainer API doesn't help, these fundamentals will save you.